# BART-large-CNN • Meta-Review Generation (LoRA)

Fine-tune `facebook/bart-large-cnn` with LoRA on ICLR 2025 review data to generate meta-reviews and accept/reject decisions.

**Runtime:** Go to *Runtime → Change runtime type* and select **T4 GPU** (or A100 if available).

In [ ]:
!pip install -q transformers datasets peft accelerate evaluate rouge-score bert-score nltk scikit-learn

In [ ]:
import json, math, random, csv
from pathlib import Path
from typing import Dict, List, Optional

import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support, classification_report,
)
from datasets import load_dataset
from transformers import (
    AutoModelForSeq2SeqLM, AutoTokenizer,
    DataCollatorForSeq2Seq, Seq2SeqTrainer,
    Seq2SeqTrainingArguments, set_seed,
)
from peft import LoraConfig, get_peft_model, TaskType, PeftModel, PeftConfig
from tqdm.auto import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch {torch.__version__}  |  Device: {device}")
if device == "cuda":
    gpu = torch.cuda.get_device_properties(0)
    print(f"GPU: {gpu.name}  |  VRAM: {gpu.total_memory / 1e9:.1f} GB")

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

# ── Point this to YOUR folder on Google Drive ──────────────
DRIVE_FOLDER = Path("/content/drive/MyDrive/meta-review-ai")
# ───────────────────────────────────────────────────────────

CSV_PATH = list(DRIVE_FOLDER.glob("*.csv"))
assert CSV_PATH, f"No CSV found in {DRIVE_FOLDER} — check the folder name above"
CSV_PATH = CSV_PATH[0]
print(f"Found dataset: {CSV_PATH.name}")

PROJECT  = Path("/content/meta-review-ai")
PROC_DIR = PROJECT / "data" / "processed"
PRED_DIR = PROJECT / "data" / "predictions"
RUNS_DIR = PROJECT / "runs"

for d in [PROC_DIR, PRED_DIR, RUNS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

## 1 — Load raw data

In [ ]:
df_raw = pd.read_csv(CSV_PATH)
print(f"Loaded {len(df_raw):,} rows  |  {df_raw['paper_id'].nunique():,} unique papers")
print(f"Columns: {list(df_raw.columns)}")
df_raw.head(2)

## 2 — Preprocess: CSV → structured JSONL

Groups reviews by paper, builds structured prompts, and creates stratified train/val/test splits (80/10/10).

In [ ]:
def safe_str(x) -> str:
    if x is None:
        return ""
    if isinstance(x, float) and math.isnan(x):
        return ""
    s = str(x).strip()
    if s.lower() in {"nan", "none"}:
        return ""
    return s

def trunc_chars(s: str, max_chars: int) -> str:
    s = " ".join(safe_str(s).split())
    if not s:
        return ""
    return s if len(s) <= max_chars else s[: max_chars - 3].rstrip() + "..."

def to_float_or_none(x):
    s = safe_str(x)
    if not s:
        return None
    try:
        return float(s)
    except ValueError:
        return None

def map_label(label: str):
    s = safe_str(label).lower()
    if s.startswith("accept"):
        return "ACCEPT"
    if s.startswith("reject"):
        return "REJECT"
    return None

TRUNC = dict(title=250, abstract=1200, summary=900,
             strengths=700, weaknesses=700, questions=500)

def build_input_text(paper_id, title, abstract, reviews_df):
    df = reviews_df.copy()
    df["_rating"] = df.get("final_rating", pd.Series([None]*len(df))).apply(to_float_or_none)
    df["_conf"]   = df.get("confidence",   pd.Series([None]*len(df))).apply(to_float_or_none)
    df = df.sort_values(
        [df["_rating"].fillna(9999.0).name, df["_conf"].fillna(-1.0).name],
        ascending=[True, False],
    )
    # re-sort properly
    df["_rs"] = df["_rating"].fillna(9999.0)
    df["_cs"] = df["_conf"].fillna(-1.0)
    df = df.sort_values(["_rs", "_cs"], ascending=[True, False])

    ratings = [x for x in df["_rating"] if x is not None]
    confs   = [x for x in df["_conf"]   if x is not None]

    parts: List[str] = []
    parts.append(
        "You are a senior ICLR meta-reviewer.\n"
        "Based on the paper and reviews, write the final meta-review and decision.\n\n"
        "You MUST follow this exact format:\n"
        "DECISION: <ACCEPT or REJECT>\n"
        "META_REVIEW:\n"
        "<your full meta-review>\n"
        "Do not output anything outside this format.\n"
    )
    parts.append(f"PAPER ID:\n{paper_id}\n")
    parts.append(f"PAPER TITLE:\n{trunc_chars(title, TRUNC['title']) or 'N/A'}\n")
    parts.append(f"ABSTRACT:\n{trunc_chars(abstract, TRUNC['abstract']) or 'N/A'}\n")

    parts.append("AGGREGATES:")
    parts.append(f"- NumReviews: {len(df)}")
    if ratings:
        parts.append(f"- MeanRating: {np.mean(ratings):.2f}")
        parts.append(f"- RatingRange: {np.min(ratings):.0f} to {np.max(ratings):.0f}")
    if confs:
        parts.append(f"- MeanConfidence: {np.mean(confs):.2f}")
    parts.append("")
    parts.append("REVIEWS:\n")

    get = lambda row, col: safe_str(row[col]) if col in row.index and pd.notna(row[col]) else ""

    for i, (_, r) in enumerate(df.iterrows(), start=1):
        parts.append(f"REVIEW {i}:")
        for field in ["final_rating", "confidence", "soundness", "presentation", "contribution"]:
            v = get(r, field)
            if v:
                parts.append(f"{field.replace('_', ' ').title().replace(' ', '')}: {v}")
        for key in ["summary", "strengths", "weaknesses", "questions"]:
            t = trunc_chars(get(r, key), TRUNC.get(key, 700))
            if t:
                parts.append(f"{key.title()}:\n{t}")
        parts.append("")

    return "\n".join(parts).strip() + "\n"

def build_target_text(decision, meta_review):
    return f"DECISION: {decision}\nMETA_REVIEW:\n{safe_str(meta_review)}\n"

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

df = df_raw.copy()
df["decision"] = df["official_label"].apply(map_label)
df = df[df["decision"].notna()].copy()
df["meta_review_clean"] = df["meta_review"].astype(str).str.strip()
df = df[df["meta_review_clean"].str.len() > 0]
df = df[df["meta_review_clean"].str.lower() != "nan"]

examples: List[Dict] = []
for paper_id, g in tqdm(df.groupby("paper_id", sort=False), desc="Building prompts"):
    title    = safe_str(g["title"].iloc[0]) if "title" in g.columns else ""
    abstract = safe_str(g["abstract"].iloc[0]) if "abstract" in g.columns else ""
    decision = safe_str(g["decision"].iloc[0])
    meta_rev = safe_str(g["meta_review_clean"].iloc[0])
    examples.append({
        "paper_id":   str(paper_id),
        "input_text":  build_input_text(str(paper_id), title, abstract, g),
        "target_text": build_target_text(decision, meta_rev),
        "decision":    decision,
    })

labels = [e["decision"] for e in examples]
idxs   = np.arange(len(examples))

train_idx, test_idx = train_test_split(
    idxs, test_size=0.10, random_state=SEED, stratify=labels)
rem_labels = [examples[i]["decision"] for i in train_idx]
train_idx, val_idx = train_test_split(
    train_idx, test_size=0.10 / 0.90, random_state=SEED, stratify=rem_labels)

def write_jsonl(path, indices):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for i in indices:
            f.write(json.dumps(examples[i], ensure_ascii=False) + "\n")

write_jsonl(PROC_DIR / "train.jsonl", train_idx)
write_jsonl(PROC_DIR / "val.jsonl",   val_idx)
write_jsonl(PROC_DIR / "test.jsonl",  test_idx)

print(f"\n\u2705 {len(train_idx)} train / {len(val_idx)} val / {len(test_idx)} test")
print(f"Labels: { {k: int(v) for k,v in pd.Series(labels).value_counts().items()} }")

## 3 — Train BART-large-CNN with LoRA

Defaults are tuned for a **T4 (16 GB)** — adjust `BATCH_SIZE` upward on A100.

In [ ]:
# ============  CONFIG  (edit here)  ============
MODEL_NAME        = "facebook/bart-large-cnn"
OUTPUT_DIR        = str(RUNS_DIR / "bart_large_cnn_run2")
TRAIN_PATH        = str(PROC_DIR / "train.jsonl")
VAL_PATH          = str(PROC_DIR / "val.jsonl")
TEST_PATH         = str(PROC_DIR / "test.jsonl")

SEED              = 42
MAX_INPUT_LEN     = 768
MAX_TARGET_LEN    = 384

NUM_EPOCHS        = 1.0
LR                = 5e-5
WEIGHT_DECAY      = 0.01
WARMUP_STEPS      = 100

BATCH_SIZE        = 2          # T4-safe; use 4–8 on A100
GRAD_ACCUM        = 8          # effective batch = BATCH_SIZE × GRAD_ACCUM
EVAL_BATCH        = 4

LOG_STEPS         = 10
EVAL_STEPS        = 200
SAVE_STEPS        = 200
SAVE_LIMIT        = 2

USE_FP16          = torch.cuda.is_available()

LORA_R            = 8
LORA_ALPHA        = 16
LORA_DROPOUT      = 0.05

GEN_MAX_TOKENS    = 256
GEN_BEAMS         = 4
# ================================================

In [ ]:
set_seed(SEED)
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

ds = load_dataset("json", data_files={"train": TRAIN_PATH, "validation": VAL_PATH})
print(ds)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

lora_cfg = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
    bias="none",
    target_modules=["q_proj", "v_proj"],
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

def preprocess(batch):
    enc = tokenizer(batch["input_text"],  max_length=MAX_INPUT_LEN,
                    truncation=True, padding=False)
    lab = tokenizer(text_target=batch["target_text"], max_length=MAX_TARGET_LEN,
                    truncation=True, padding=False)
    enc["labels"] = lab["input_ids"]
    return enc

tok_ds = ds.map(preprocess, batched=True,
                remove_columns=ds["train"].column_names, desc="Tokenizing")

collator = DataCollatorForSeq2Seq(
    tokenizer, model=model, padding="longest", label_pad_token_id=-100)

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    seed=SEED,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LR,
    warmup_steps=WARMUP_STEPS,
    weight_decay=WEIGHT_DECAY,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=SAVE_LIMIT,
    logging_steps=LOG_STEPS,
    report_to="none",
    predict_with_generate=True,
    generation_max_length=GEN_MAX_TOKENS,
    generation_num_beams=GEN_BEAMS,
    fp16=USE_FP16,
    dataloader_pin_memory=True,
)

trainer = Seq2SeqTrainer(
    model=model, args=training_args,
    train_dataset=tok_ds["train"],
    eval_dataset=tok_ds["validation"],
    data_collator=collator,
)

result = trainer.train()

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
model.config.save_pretrained(OUTPUT_DIR)

metrics = result.metrics
with open(Path(OUTPUT_DIR) / "train_metrics.json", "w") as f:
    json.dump({k: (float(v) if isinstance(v, np.floating) else v)
               for k, v in metrics.items()}, f, indent=2)

print(f"\n\u2705 Training complete  \u2014  saved to {OUTPUT_DIR}")
print(f"Loss: {metrics.get('train_loss', 'N/A')}")

## 4 — Generate predictions on the test set

In [ ]:
PRED_PATH = str(PRED_DIR / "bart_run2_predictions.csv")

peft_conf  = PeftConfig.from_pretrained(OUTPUT_DIR)
tok_infer  = AutoTokenizer.from_pretrained(peft_conf.base_model_name_or_path)
base_model = AutoModelForSeq2SeqLM.from_pretrained(peft_conf.base_model_name_or_path)
infer_model = PeftModel.from_pretrained(base_model, OUTPUT_DIR)
infer_model = infer_model.merge_and_unload()
infer_model.to(device).eval()

def extract_decision(text):
    text = (text or "").strip()
    if "DECISION:" in text:
        line = text.split("DECISION:", 1)[1].strip().split("\n", 1)[0].upper()
        if line.startswith("ACCEPT"):
            return "ACCEPT"
        if line.startswith("REJECT"):
            return "REJECT"
        return line
    return "UNKNOWN"

rows = []
with open(TEST_PATH, encoding="utf-8") as f:
    lines = f.readlines()

for line in tqdm(lines, desc="Predicting"):
    ex = json.loads(line)
    inputs = tok_infer(
        ex["input_text"], return_tensors="pt",
        truncation=True, max_length=MAX_INPUT_LEN,
    ).to(device)
    with torch.no_grad():
        out = infer_model.generate(
            **inputs, max_new_tokens=GEN_MAX_TOKENS, num_beams=GEN_BEAMS)
    pred = tok_infer.decode(out[0], skip_special_tokens=True)
    rows.append({
        "paper_id":         ex["paper_id"],
        "true_decision":    extract_decision(ex["target_text"]),
        "pred_decision":    extract_decision(pred),
        "true_meta_review": ex["target_text"],
        "pred_meta_review": pred,
    })

with open(PRED_PATH, "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=rows[0].keys())
    w.writeheader()
    w.writerows(rows)

print(f"\u2705 {len(rows)} predictions saved \u2192 {PRED_PATH}")

## 5 — Evaluate

In [ ]:
import evaluate

pred_df = pd.read_csv(PRED_PATH)
y_true  = pred_df["true_decision"].tolist()
y_pred  = pred_df["pred_decision"].tolist()

acc = accuracy_score(y_true, y_pred)
prec, rec, f1, _ = precision_recall_fscore_support(
    y_true, y_pred, average="macro", zero_division=0)

print("=" * 50)
print("DECISION CLASSIFICATION")
print("=" * 50)
print(f"Accuracy:  {acc:.4f}")
print(f"Macro P:   {prec:.4f}")
print(f"Macro R:   {rec:.4f}")
print(f"Macro F1:  {f1:.4f}")
print()
print(classification_report(y_true, y_pred, zero_division=0))

refs  = pred_df["true_meta_review"].tolist()
preds = pred_df["pred_meta_review"].tolist()

rouge = evaluate.load("rouge")
rouge_scores = rouge.compute(predictions=preds, references=refs)

print("=" * 50)
print("META-REVIEW TEXT  (ROUGE)")
print("=" * 50)
for k, v in rouge_scores.items():
    print(f"  {k:12s}: {v:.4f}")

bertscore = evaluate.load("bertscore")
bs = bertscore.compute(predictions=preds, references=refs, lang="en", batch_size=32)
bs_f1 = float(np.mean(bs["f1"]))
print(f"\n  {'BERTScore-F1':12s}: {bs_f1:.4f}")

all_metrics = {
    "accuracy": acc, "precision_macro": prec,
    "recall_macro": rec, "f1_macro": f1,
    **rouge_scores,
    "bertscore_precision": float(np.mean(bs["precision"])),
    "bertscore_recall":    float(np.mean(bs["recall"])),
    "bertscore_f1":        bs_f1,
}

metrics_file = PRED_DIR / "bart_run2_eval_metrics.json"
with open(metrics_file, "w") as f:
    json.dump(all_metrics, f, indent=2)
print(f"\n\u2705 Metrics saved \u2192 {metrics_file}")

## 6 — Back up to Google Drive

In [ ]:
import shutil

dest = DRIVE_FOLDER / "bart_large_cnn_run2"
if dest.exists():
    shutil.rmtree(dest)
shutil.copytree(OUTPUT_DIR, str(dest))

shutil.copy2(PRED_PATH, str(DRIVE_FOLDER / "bart_run2_predictions.csv"))
shutil.copy2(str(metrics_file), str(DRIVE_FOLDER / "bart_run2_eval_metrics.json"))

print(f"\u2705 Backed up to {DRIVE_FOLDER}")
!ls -lh "{DRIVE_FOLDER}"